# 04. Deduplication and Record Linkage Basics
**Engineering Data Processing with Python** | one-day course | about 30 minutes

> **TRAINER / SOLUTION VERSION.** Every blank is filled in and the notebook has been run.


In [1]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## The point of this module

Duplicates inflate counts and costs. Deleting them carelessly removes real records. You will:

1. Decide whether duplicates are mistakes at all.
2. Remove exact copies.
3. Handle the same work order exported twice with different details, using a rule you can explain.
4. Send conflicts you cannot resolve to a person.
5. Match technician names that are spelt differently.

We start from a trainer copy of Module 03's clean outputs, so everyone has the same data.

In [2]:
DATES = ["date_raised", "date_closed", "last_modified"]
wo = pd.read_csv("data/checkpoints/m03_work_orders_clean.csv", parse_dates=DATES)
wo["priority"] = wo["priority"].astype("Int64")
reg = pd.read_csv("data/checkpoints/m03_asset_register_clean.csv", parse_dates=["install_date"])
print(wo.shape, reg.shape)

(1016, 13) (62, 10)


## 1. Are duplicates always mistakes?

Example: find the duplicate rows in the concrete test data and look at a few.

In [3]:
concrete = pd.read_csv("data/concrete.csv")
print("Duplicate rows:", concrete.duplicated().sum())
concrete[concrete.duplicated(keep=False)].sort_values(list(concrete.columns)).head(4)

Duplicate rows: 25


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
801,252.0,0.0,0.0,185.0,0.0,1111.0,784.0,28,19.691435
809,252.0,0.0,0.0,185.0,0.0,1111.0,784.0,28,19.691435
83,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171
86,362.6,189.0,0.0,164.9,11.6,944.7,755.8,3,35.301171


Same mix, same age, and strength identical to 8 decimal places. Genuine repeat cubes would not give identical
results to 8 decimal places, so these look like copy errors. The key point: you **looked and decided**, and you
can say why.

### Your turn 4.1
Remove the duplicates. Choose from: `duplicated`, `drop_duplicates`.

In [4]:
concrete_clean = concrete.drop_duplicates()
print(len(concrete), "->", len(concrete_clean), "rows")

1030 -> 1005 rows


## 2. Exact copies in the work orders

### Your turn 4.2
Same pattern on the work orders.

In [5]:
print("Exact duplicate rows:", wo.duplicated().sum())
wo = wo.drop_duplicates()
print("Rows now:", len(wo))

Exact duplicate rows: 18
Rows now: 998


## 3. Same work order, different details

Every `wo_id` should appear once. Some still appear twice. Run and read:

In [6]:
print("Every wo_id unique?", wo["wo_id"].is_unique)
repeats = wo[wo["wo_id"].duplicated(keep=False)].sort_values(["wo_id", "last_modified"])
repeats[["wo_id", "status", "date_closed", "last_modified"]].head(6)

Every wo_id unique? False


,wo_id,status,date_closed,last_modified
285,WO-25-00013,Open,NaT,2025-01-12 19:51:06
784,WO-25-00013,Closed,2025-01-15 17:52:00,2025-01-15 19:51:06
392,WO-25-00039,Open,NaT,2025-01-27 23:29:15
664,WO-25-00039,Closed,2025-01-31 22:18:00,2025-01-31 23:29:15
914,WO-25-00049,Open,NaT,2025-02-04 14:10:44
93,WO-25-00049,Closed,2025-02-07 11:29:00,2025-02-07 14:10:44


Each pair is the same job exported twice: once while **Open**, again after it was **Closed**.
`last_modified` shows which version is newer.

**Rule:** sort by `last_modified` (oldest first), then for each `wo_id` keep the **last** row, the newest.

### Your turn 4.3
Should `keep` be `"first"` or `"last"`?

In [7]:
wo = wo.sort_values("last_modified").drop_duplicates(subset="wo_id", keep="last")
print("Every wo_id unique?", wo["wo_id"].is_unique, "| rows:", len(wo))

Every wo_id unique? True | rows: 973


In [8]:
kept = wo.set_index("wo_id").loc[repeats["wo_id"].unique(), "status"]
check((kept == "Closed").all(), "Kept the newer, Closed version of every repeated job.",
      "You kept the older Open versions. Which end of a sorted list is the newest?")

PASS: Kept the newer, Closed version of every repeated job.


"Keep the first one" without sorting is not a rule. It depends on whatever order the rows happened to be in.

## 4. Conflicts a computer should not decide

The asset register should list each asset once. Run and read:

In [9]:
reg = reg.drop_duplicates()                                   # exact copies first
conflicts = reg[reg["asset_id"].duplicated(keep=False)]
conflicts[["asset_id", "asset_type", "site", "manufacturer"]]

,asset_id,asset_type,site,manufacturer
9,CNC-010,CNC Mill,Cork,Hartwell Machine Tools
61,CNC-010,CNC Mill,Limerick,Hartwell Machine Tools


Same asset, two different sites. Was it moved? Is one a typo? The data cannot tell you. A person can.
So we save it for review, flag it, and keep the first row for now.

In [10]:
conflicts.to_csv("outputs/asset_conflicts_for_review.csv", index=False)
reg["needs_review"] = reg["asset_id"].isin(conflicts["asset_id"])
reg = reg.drop_duplicates(subset="asset_id", keep="first")
print(len(reg), "assets, flagged for review:", reg["needs_review"].sum())

60 assets, flagged for review: 1


## 5. Technician names spelt differently

Eight technicians, about 30 spellings: capitals, missing fadas (Ciarán / Ciaran), initials, "Murphy, Declan".

In [11]:
wo["technician"].value_counts().head(15)

technician
Seán Kelly            93
Declan Murphy         91
Orla Doyle            85
Niamh O'Connor        82
Gráinne Fitzgerald    80
Aoife Byrne           79
Ciarán Walsh          69
Patrick Ryan          68
O. Doyle              23
aoife byrne           20
Ciaran Walsh          19
declan murphy         18
Niamh O Connor        17
Murphy, Declan        17
D. Murphy             17
Name: count, dtype: int64

The helper below does two things for you:
1. **Normalises** each name: lower case, fadas removed, "Murphy, Declan" flipped to "declan murphy".
2. **Fuzzy matches** it against the official staff list: "is this spelling very close to a real name?"

Run it and read what it could and could not match.

In [12]:
# @title Helper: normalise and fuzzy-match technician names (just run it)
import unicodedata, difflib

STAFF = ["Aoife Byrne", "Ciarán Walsh", "Declan Murphy", "Niamh O'Connor",
         "Seán Kelly", "Orla Doyle", "Patrick Ryan", "Gráinne Fitzgerald"]

def normalise_name(name):
    if pd.isna(name):
        return name
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    text = text.replace("'", "").replace(".", " ")
    if "," in text:
        last, first = [p.strip() for p in text.split(",", 1)]
        text = f"{first} {last}"
    return " ".join(text.lower().split())

staff_by_norm = {normalise_name(s): s for s in STAFF}
wo["tech_norm"] = wo["technician"].apply(normalise_name)

auto_map = {}
for spelling in wo["tech_norm"].dropna().unique():
    hit = difflib.get_close_matches(spelling, staff_by_norm.keys(), n=1, cutoff=0.85)
    if hit:
        auto_map[spelling] = staff_by_norm[hit[0]]

unmatched = sorted(set(wo["tech_norm"].dropna()) - set(auto_map))
print("Spellings matched automatically:", len(auto_map))
print("Need a human:", unmatched)

Spellings matched automatically: 9
Need a human: ['a byrne', 'c walsh', 'd murphy', 'g fitzgerald', 'n oconnor', 'o doyle', 'pat ryan']


Initials (`d murphy`) and nicknames (`pat ryan`) are not close enough to match automatically. That is correct:
if Kilbrack hired a Deirdre Murphy, `d murphy` would be ambiguous. Someone who knows the team decides.

### Your turn 4.4
You know the team. Fill in the last two: `d murphy` is **Declan Murphy** and `pat ryan` is **Patrick Ryan**.

In [13]:
MANUAL_MAP = {
    "a byrne": "Aoife Byrne",
    "c walsh": "Ciarán Walsh",
    "g fitzgerald": "Gráinne Fitzgerald",
    "n oconnor": "Niamh O'Connor",
    "o doyle": "Orla Doyle",
    "d murphy": "Declan Murphy",
    "pat ryan": "Patrick Ryan",
}
name_map = {**auto_map, **MANUAL_MAP}          # combine both lookups

wo["technician_raw"] = wo["technician"]
wo["technician"] = wo["tech_norm"].map(name_map)
wo["technician"].value_counts(dropna=False)

technician
Declan Murphy         143
Seán Kelly            130
Orla Doyle            128
Gráinne Fitzgerald    119
Aoife Byrne           117
Niamh O'Connor        115
Ciarán Walsh          114
Patrick Ryan          107
Name: count, dtype: int64

In [14]:
check(wo["technician"].isna().sum() == 0 and wo["technician"].nunique() == 8,
      "Every work order now has one of the 8 official names.", "Some names are still blank. Check the spelling.")

PASS: Every work order now has one of the 8 official names.


Save the mapping so someone can check it and reuse it next month:

In [15]:
tech_map = wo[["technician_raw", "tech_norm", "technician"]].drop_duplicates()
tech_map["method"] = np.where(tech_map["tech_norm"].isin(MANUAL_MAP.keys()), "manual", "fuzzy")
tech_map.to_csv("outputs/technician_map.csv", index=False)

wo = wo.drop(columns="tech_norm").sort_values("date_raised")
wo.to_csv("outputs/work_orders_dedup.csv", index=False)
reg.to_csv("outputs/asset_register_dedup.csv", index=False)
print("Saved", wo.shape, reg.shape)

Saved (973, 14) (60, 11)


## Recap
* Look at duplicates before deleting them, and write down why.
* Exact copies: `drop_duplicates()`. Same ID, different details: sort by a date and keep the newest.
* When rows disagree and the data cannot say which is right, flag it and ask a person.
* Fuzzy matching suggests; a person approves. Save the mapping.

**Optional, with Gemini:** ask *"What could go wrong if I used fuzzy matching with a low cutoff like 0.6 on people's
names?"* Do you agree with its answer?

Next: **05 Combining Data from Multiple Sources**.